In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# PATH = "../.."
PATH = "storage"

# If required:
github_path = os.path.join(PATH, "third_party", "EXAONEPath")
sys.path.append(github_path)
from vision_transformer import VisionTransformer
import timm
from torchvision import transforms, models
from timm.layers import SwiGLUPacked

# If using HuggingFace
from transformers import AutoModel, AutoImageProcessor
from vision_transformer import VisionTransformer
import pfm_robustness.models


In [2]:
class ProcessorTransform:
    def __init__(self, processor):
        self.processor = processor
    def __call__(self, img):
        return self.processor(img, return_tensors="pt")["pixel_values"].squeeze(0)

uni_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
giga_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
exaone_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
hoptimus_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.707223, 0.578729, 0.703617), std=(0.211883, 0.230117, 0.177517))
])
virchow_transform = transforms.Compose([
    transforms.Resize(size=224, interpolation=transforms.InterpolationMode.BICUBIC, max_size=None, antialias=True),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4850, 0.4560, 0.4060), std=(0.2290, 0.2240, 0.2250))
])

# HuggingFace processor transforms
resnet_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-18")
hibou_b_processor = AutoImageProcessor.from_pretrained("histai/hibou-b", trust_remote_code=True)
hibou_L_processor = AutoImageProcessor.from_pretrained("histai/hibou-L", trust_remote_code=True)
phikon_processor = AutoImageProcessor.from_pretrained("owkin/phikon", trust_remote_code=True)
phikonv2_processor = AutoImageProcessor.from_pretrained("owkin/phikon-v2", trust_remote_code=True)

resnet_transform = ProcessorTransform(resnet_processor)
hibou_b_transform = ProcessorTransform(hibou_b_processor)
hibou_L_transform = ProcessorTransform(hibou_L_processor)
phikon_transform = ProcessorTransform(phikon_processor)
phikonv2_transform = ProcessorTransform(phikonv2_processor)

model_transforms = {
    "ResNet18":     resnet_transform,
    "ResNet50":     resnet_transform,
    "UNI":          uni_transform,
    "UNI2":         uni_transform,
    "GigaPath":     giga_transform,
    "Virchow":      virchow_transform,
    "Virchow2":     virchow_transform,
    "EXAONEPath":   exaone_transform,
    "Hibou-B":      hibou_b_transform,
    "Hibou-L":      hibou_L_transform,
    "H-Optimus-0":  hoptimus_transform,
    "H-Optimus-1":  hoptimus_transform,
    "H0-mini":      hoptimus_transform,
    "Phikon v1":    phikon_transform,
    "Phikon v2":    phikonv2_transform,
}

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [3]:
img_dir = os.path.join(PATH, "data", "PatchCamelyon", "train_images_sampled")
patches = []
for fname in os.listdir(img_dir):
    if fname.endswith(".png"):
        name_parts = fname.replace(".png", "").split("_")
        # Adjust if WSI naming is different!
        wsi = "_".join(name_parts[:2])  # tumor_003, normal_018, etc.
        label = 1 if name_parts[-1] == "TRUE" else 0
        patches.append({"img_path": os.path.join(img_dir, fname), "wsi": wsi, "label": label})
df_patches = pd.DataFrame(patches)

In [4]:
class PatchDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["img_path"]).convert("RGB")
        image = self.transform(image)
        label = row["label"]
        return image, label

In [ ]:
# FOLD PARAMETERS
n_folds = 5
n_epochs = 10
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"
cluster_dir = os.path.join(PATH, "data", "PatchCamelyon", "clusters")

# List your model loader functions (must match pfm_robustness.models naming convention)
model_loaders = {
    "ResNet18": pfm_robustness.models.load_resnet18,
    "ResNet50": pfm_robustness.models.load_resnet50,
    "UNI": pfm_robustness.models.load_uni,
    "UNI2": pfm_robustness.models.load_uni2,
    "GigaPath": pfm_robustness.models.load_gigapath,
    "Virchow": pfm_robustness.models.load_virchow,
    "Virchow2": pfm_robustness.models.load_virchow2,
    "EXAONEPath": pfm_robustness.models.load_exaonepath,
    "Hibou-B": pfm_robustness.models.load_hibou_b,
    "Hibou-L": pfm_robustness.models.load_hibou_l,
    "Phikon v1": pfm_robustness.models.load_phikon,
    "Phikon v2": pfm_robustness.models.load_phikon_v2,
    "H-Optimus-0": pfm_robustness.models.load_h_optimus0,
    "H-Optimus-1": pfm_robustness.models.load_h_optimus1,
    "H0-mini": pfm_robustness.models.load_h0_mini,
}

all_results = {}

for model_name, model_loader in model_loaders.items():
    print(f"\n===== {model_name} =====")
    if model_name not in model_transforms:
        print(f"Skipping {model_name}: no transform defined.")
        continue

    # Load cluster assignments for this model
    fold_csv = os.path.join(cluster_dir, f"kmeans_folds_{model_name}.csv")
    if not os.path.exists(fold_csv):
        print(f"Skipping {model_name}: missing cluster CSV.")
        continue

    fold_df = pd.read_csv(fold_csv)
    wsi_to_fold = dict(zip(fold_df["wsi"], fold_df["fold"]))
    df_patches["fold"] = df_patches["wsi"].map(wsi_to_fold)
    if df_patches["fold"].isnull().any():
        print(f"Warning: Not all patches assigned a fold for {model_name}. Skipping.")
        continue

    fold_metrics = []

    for fold in range(n_folds):
        print(f"\nFold {fold+1}/{n_folds}")
        # Train on all except fold, test on fold
        train_df = df_patches[df_patches["fold"] != fold]
        test_df = df_patches[df_patches["fold"] == fold]

        train_dataset = PatchDataset(train_df, model_transforms[model_name])
        test_dataset = PatchDataset(test_df, model_transforms[model_name])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

        # Model: reload and reinit for each fold (head only)
        model, optimizer = model_loader(n_classes=2, device=device)

        criterion = nn.NLLLoss()
        model.train()
        for epoch in range(n_epochs):
            n_batches = len(train_loader)
            checkpoints = set([int(n_batches * p / 5) for p in range(1, 6)])  # 20%,40%,60%,80%,100%
            for i, (images, labels) in enumerate(train_loader):
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                if (i + 1) in checkpoints:
                    percent = int(100 * (i + 1) / n_batches)
                    print(f"    Epoch {epoch+1}/{n_epochs} - {percent}% complete (batch {i+1}/{n_batches})")
        # --- Evaluation ---
        model.eval()
        preds, targs = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                preds.extend(outputs.argmax(dim=1).cpu().numpy())
                targs.extend(labels.cpu().numpy())
        acc = accuracy_score(targs, preds)
        f1 = f1_score(targs, preds)
        cm = confusion_matrix(targs, preds)
        try:
            auc = roc_auc_score(targs, preds)
        except Exception:
            auc = float('nan')
        fold_metrics.append({"accuracy": acc, "f1": f1, "auc": auc, "confusion": cm})

        print(f"  Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {auc:.3f}")

    # Summarize
    all_accs = [m["accuracy"] for m in fold_metrics]
    all_f1s = [m["f1"] for m in fold_metrics]
    all_aucs = [m["auc"] for m in fold_metrics]
    print(f"\n[{model_name}] Mean ± Std over {n_folds} folds:")
    print(f"  Accuracy: {np.mean(all_accs):.3f} ± {np.std(all_accs):.3f}")
    print(f"  F1:       {np.mean(all_f1s):.3f} ± {np.std(all_f1s):.3f}")
    print(f"  ROC-AUC:  {np.nanmean(all_aucs):.3f} ± {np.nanstd(all_aucs):.3f}")

    all_results[model_name] = fold_metrics


===== ResNet18 =====

Fold 1/5


c:\Users\Dhyey\anaconda3\envs\cs907\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Dhyey\anaconda3\envs\cs907\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
import pandas as pd
import numpy as np

results_rows = []

for model_name, fold_metrics in all_results.items():
    for fold, m in enumerate(fold_metrics):
        results_rows.append({
            "model": model_name,
            "fold": fold + 1,
            "accuracy": m["accuracy"],
            "f1": m["f1"],
            "auc": m["auc"],
            "confusion_matrix": str(m["confusion"])
        })
    # Add summary stats (mean/std for this model)
    all_accs = [m["accuracy"] for m in fold_metrics]
    all_f1s = [m["f1"] for m in fold_metrics]
    all_aucs = [m["auc"] for m in fold_metrics]
    results_rows.append({
        "model": model_name,
        "fold": "mean",
        "accuracy": np.mean(all_accs),
        "f1": np.mean(all_f1s),
        "auc": np.nanmean(all_aucs),
        "confusion_matrix": "N/A"
    })
    results_rows.append({
        "model": model_name,
        "fold": "std",
        "accuracy": np.std(all_accs),
        "f1": np.std(all_f1s),
        "auc": np.nanstd(all_aucs),
        "confusion_matrix": "N/A"
    })

# Create dataframe and save
df_results = pd.DataFrame(results_rows)
csv_path = os.path.join(PATH, "data", "PatchCamelyon", "model_fold_metrics.csv")
df_results.to_csv(csv_path, index=False)
print(f"\nSaved full per-fold and summary metrics to:\n  {csv_path}")

# Optionally display
display(df_results)